# Explainable Credit Risk Prediction Using Machine Learning

CS465 Machine Learning Course Project

This notebook implements a complete end-to-end pipeline for credit risk prediction on the UCI German Credit Dataset.

## A. Imports and Setup

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from pathlib import Path

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, classification_report,
                           confusion_matrix, roc_curve, balanced_accuracy_score)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Explainability
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available, will skip SHAP analysis")

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Suppress warnings
warnings.filterwarnings('ignore')

# Set up matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Set up robust paths using pathlib with explicit debugging
cwd = Path.cwd()
print("Current working directory:", cwd)

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "german.data"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_PATH:", DATA_PATH.resolve())
print("FIGURES_DIR:", FIGURES_DIR.resolve())
print("TABLES_DIR:", TABLES_DIR.resolve())
print("METRICS_DIR:", METRICS_DIR.resolve())

print("Setup complete!")
print(f"Random seed set to: {RANDOM_SEED}")
print(f"Output directories created")

: 

# Verification: Test figure saving
test_path = FIGURES_DIR / "test_figure.png"

plt.figure()
plt.plot([1, 2, 3], [1, 4, 9])
plt.title("Test Figure")
plt.savefig(test_path, dpi=300, bbox_inches="tight")
plt.close()

print("Test figure saved to:", test_path.resolve())
print("Current files in figures directory:")
for f in FIGURES_DIR.glob("*"):
    print(" -", f.name)

In [ ]:
# Define column names as specified
column_names = [
    "status_checking_account",
    "duration_months",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate",
    "personal_status_sex",
    "other_debtors_guarantors",
    "present_residence_since",
    "property",
    "age_years",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "num_dependents",
    "telephone",
    "foreign_worker",
    "target"
]

# Load the dataset
try:
    df = pd.read_csv(
        DATA_PATH,
        sep=r"\s+", 
        header=None, 
        names=column_names
    )
    print("Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print("Error: Could not find the dataset file.")
    print(f"Please ensure the dataset exists at: {DATA_PATH}")
    raise

# Map target values: 1 -> 0 (good credit), 2 -> 1 (bad credit)
df['target'] = df['target'].map({1: 0, 2: 1})

# Add target mapping assertion
assert df['target'].isin([0, 1]).all(), "Target mapping failed."

print("Target mapping applied: 1->0 (good), 2->1 (bad)")
print(f"Target distribution after mapping:")
print(df['target'].value_counts().sort_index())

## C. Dataset Overview

In [ ]:
# Display basic information
print("=== DATASET SHAPE ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print()

print("=== FIRST 5 ROWS ===")
display(df.head())
print()

print("=== DATA TYPES ===")
print(df.dtypes)
print()

print("=== MISSING VALUES ===")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.any() else "No missing values found")
print()

print("=== DUPLICATE ROWS ===")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
print()

print("=== TARGET DISTRIBUTION ===")
target_counts = df['target'].value_counts()
target_percentages = df['target'].value_counts(normalize=True) * 100
for target_val in sorted(target_counts.index):
    label = "Good Credit (0)" if target_val == 0 else "Bad Credit (1)"
    print(f"{label}: {target_counts[target_val]} ({target_percentages[target_val]:.1f}%)")

## D. Exploratory Data Analysis

In [ ]:
# Target class distribution plot
plt.figure(figsize=(8, 6))
target_counts = df['target'].value_counts().sort_index()
labels = ['Good Credit (0)', 'Bad Credit (1)']
colors = ['lightgreen', 'lightcoral']

bars = plt.bar(labels, target_counts.values, color=colors)
plt.title('Target Class Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.ylim(0, max(target_counts.values) * 1.1)

# Add count labels on bars
for bar, count in zip(bars, target_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Target distribution plot saved to {FIGURES_DIR / 'target_distribution.png'}")

In [ ]:
# Target class distribution plot
plt.figure(figsize=(8, 6))
target_counts = df['target'].value_counts().sort_index()
labels = ['Good Credit (0)', 'Bad Credit (1)']
colors = ['lightgreen', 'lightcoral']

bars = plt.bar(labels, target_counts.values, color=colors)
plt.title('Target Class Distribution', fontsize=14, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.ylim(0, max(target_counts.values) * 1.1)

# Add count labels on bars
for bar, count in zip(bars, target_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(count), ha='center', va='bottom', fontweight='bold')

output_path = FIGURES_DIR / "target_distribution.png"
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

In [ ]:
# Histograms for numeric columns
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(numeric_columns):
    if i < len(axes):
        axes[i].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        axes[i].set_title(f'{col.replace("_", " ").title()}', fontweight='bold')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')
        axes[i].grid(True, alpha=0.3)

# Hide any unused subplots
for i in range(len(numeric_columns), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'numeric_histograms.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Numeric histograms saved to {FIGURES_DIR / 'numeric_histograms.png'}")

In [ ]:
# Histograms for numeric columns
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, col in enumerate(numeric_columns):
    if i < len(axes):
        axes[i].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        axes[i].set_title(f'{col.replace("_", " ").title()}', fontweight='bold')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')
        axes[i].grid(True, alpha=0.3)

# Hide any unused subplots
for i in range(len(numeric_columns), len(axes)):
    axes[i].set_visible(False)

output_path = FIGURES_DIR / 'numeric_histograms.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

In [ ]:
# Boxplots of key numeric features by target
key_numeric_features = ['duration_months', 'credit_amount', 'age_years', 'installment_rate']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(key_numeric_features):
    if feature in df.columns:
        good_credit = df[df['target'] == 0][feature]
        bad_credit = df[df['target'] == 1][feature]
        
        axes[i].boxplot([good_credit, bad_credit], labels=['Good Credit', 'Bad Credit'])
        axes[i].set_title(f'{feature.replace("_", " ").title()} by Credit Risk', fontweight='bold')
        axes[i].set_ylabel('Value')
        axes[i].grid(True, alpha=0.3)

output_path = FIGURES_DIR / 'boxplots_by_target.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(12, 8))

# Calculate correlation matrix for numeric features only
correlation_matrix = df[numeric_columns].corr()

# Create heatmap
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f',
            cbar_kws={'shrink': 0.8})

plt.title('Correlation Matrix of Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Correlation heatmap saved to {FIGURES_DIR / 'correlation_heatmap.png'}")

# Correlation heatmap for numeric features
plt.figure(figsize=(12, 8))

# Calculate correlation matrix for numeric features only
correlation_matrix = df[numeric_columns].corr()

# Create heatmap
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f',
            cbar_kws={'shrink': 0.8})

plt.title('Correlation Matrix of Numeric Features', fontsize=14, fontweight='bold')

output_path = FIGURES_DIR / 'correlation_heatmap.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

In [ ]:
# Identify categorical and numerical columns
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('target')  # Remove target from features

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")

# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_SEED, 
    stratify=y
)

print("=== TRAIN-TEST SPLIT ===")
print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTest target distribution:")
print(y_test.value_counts().sort_index())

In [ ]:
# Build preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Create column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created:")
print(f"- Numeric features: StandardScaler")
print(f"- Categorical features: OneHotEncoder (handle_unknown='ignore')")
print(f"Total features before preprocessing: {X.shape[1]}")

## F. Models

In [ ]:
# Define models with reasonable hyperparameters
models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000,
        class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=RANDOM_SEED,
        class_weight='balanced',
        max_depth=10
    ),
    'Random Forest': RandomForestClassifier(
        random_state=RANDOM_SEED,
        class_weight='balanced',
        n_estimators=100,
        max_depth=10
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=RANDOM_SEED,
        n_estimators=100,
        max_depth=6
    )
}

print("Models defined:")
for name, model in models.items():
    print(f"- {name}: {model.__class__.__name__}")

In [ ]:
# Train models and create pipelines
trained_models = {}
training_results = {}

print("=== TRAINING MODELS ===")

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Create full pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train the model
    pipeline.fit(X_train, y_train)
    
    # Store the trained pipeline
    trained_models[name] = pipeline
    
    # Get training score
    train_score = pipeline.score(X_train, y_train)
    training_results[name] = train_score
    
    print(f"{name} training accuracy: {train_score:.4f}")

print("\nAll models trained successfully!")

## G. Evaluation

In [ ]:
# Evaluate models on test set
evaluation_results = []
confusion_matrices = {}
roc_curves = {}

print("=== MODEL EVALUATION ===")

for name, pipeline in trained_models.items():
    print(f"\nEvaluating {name}...")
    
    # Make predictions
    y_pred = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]  # Probability of positive class
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    evaluation_results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Balanced Accuracy': balanced_acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    # Store confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    confusion_matrices[name] = cm
    
    # Store ROC curve data
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
    
    # Print metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")
    
    # Print classification report
    print(f"\nClassification Report:")
    report = classification_report(y_test, y_pred, target_names=['Good Credit', 'Bad Credit'])
    print(report)
    
    # Save classification report to CSV
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=['Good Credit', 'Bad Credit'],
        output_dict=True
    )
    report_df = pd.DataFrame(report_dict).transpose()
    safe_name = name.lower().replace(' ', '_')
    report_df.to_csv(METRICS_DIR / f"{safe_name}_classification_report.csv")
    print(f"Classification report saved to {METRICS_DIR / f'{safe_name}_classification_report.csv'}")

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(evaluation_results)
comparison_df = comparison_df.round(4)

print("=== MODEL COMPARISON TABLE ===")
display(comparison_df)

# Save comparison table
comparison_df.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)
comparison_df.to_csv(TABLES_DIR / 'model_comparison.csv', index=False)
print(f"\nModel comparison saved to {METRICS_DIR / 'model_comparison.csv'}")

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    if i < len(axes):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                   xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
        axes[i].set_title(f'{name} Confusion Matrix', fontweight='bold')
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrices saved to {FIGURES_DIR / 'confusion_matrices.png'}")

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    if i < len(axes):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                   xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
        axes[i].set_title(f'{name} Confusion Matrix', fontweight='bold')
        axes[i].set_xlabel('Predicted')
        axes[i].set_ylabel('Actual')

output_path = FIGURES_DIR / 'confusion_matrices.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

In [ ]:
# Plot ROC curves comparison
plt.figure(figsize=(10, 8))

for name, roc_data in roc_curves.items():
    plt.plot(roc_data['fpr'], roc_data['tpr'], 
             label=f'{name} (AUC = {roc_data["auc"]:.3f})', 
             linewidth=2)

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])

output_path = FIGURES_DIR / 'roc_curves.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

## H. Explainability

In [ ]:
# Feature importance for Random Forest
rf_pipeline = trained_models['Random Forest']
rf_model = rf_pipeline.named_steps['classifier']

# Get feature names after preprocessing
feature_names = (numerical_features + 
                 rf_pipeline.named_steps['preprocessor']
                 .named_transformers_['cat']
                 .named_steps['onehot']
                 .get_feature_names_out(categorical_features).tolist())

# Get feature importances
importances = rf_model.feature_importances_

# Create dataframe of feature importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 15 most important features
plt.figure(figsize=(12, 8))
top_features = feature_importance_df.head(15)

plt.barh(range(len(top_features)), top_features['importance'], color='skyblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Random Forest - Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # To show most important at top
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'random_forest_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Random Forest feature importance plot saved to {FIGURES_DIR / 'random_forest_feature_importance.png'}")
print("\nTop 10 most important features:")
display(feature_importance_df.head(10))

# Save feature importance data
feature_importance_df.to_csv(METRICS_DIR / 'random_forest_feature_importance.csv', index=False)

In [ ]:
# Feature importance for Random Forest
rf_pipeline = trained_models['Random Forest']
rf_model = rf_pipeline.named_steps['classifier']

# Get feature names after preprocessing
feature_names = (numerical_features + 
                 rf_pipeline.named_steps['preprocessor']
                 .named_transformers_['cat']
                 .named_steps['onehot']
                 .get_feature_names_out(categorical_features).tolist())

# Get feature importances
importances = rf_model.feature_importances_

# Create dataframe of feature importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 15 most important features
plt.figure(figsize=(12, 8))
top_features = feature_importance_df.head(15)

plt.barh(range(len(top_features)), top_features['importance'], color='skyblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance', fontsize=12)
plt.title('Random Forest - Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # To show most important at top
plt.grid(True, alpha=0.3)

output_path = FIGURES_DIR / 'random_forest_feature_importance.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

print("Top 10 most important features:")
display(feature_importance_df.head(10))

# Save feature importance data
feature_importance_df.to_csv(METRICS_DIR / 'random_forest_feature_importance.csv', index=False)

In [ ]:
# Coefficient importance for Logistic Regression
lr_pipeline = trained_models['Logistic Regression']
lr_model = lr_pipeline.named_steps['classifier']

# Get feature names (same as Random Forest)
lr_feature_names = (numerical_features + 
                     lr_pipeline.named_steps['preprocessor']
                     .named_transformers_['cat']
                     .named_steps['onehot']
                     .get_feature_names_out(categorical_features).tolist())

# Get coefficients
coefficients = lr_model.coef_[0]

# Create dataframe of coefficients
coefficient_df = pd.DataFrame({
    'feature': lr_feature_names,
    'coefficient': coefficients,
    'abs_coefficient': np.abs(coefficients)
}).sort_values('abs_coefficient', ascending=False)

# Plot top 15 most influential features (by absolute coefficient)
plt.figure(figsize=(12, 8))
top_coefficients = coefficient_df.head(15)

colors = ['red' if x < 0 else 'green' for x in top_coefficients['coefficient']]
plt.barh(range(len(top_coefficients)), top_coefficients['abs_coefficient'], color=colors)
plt.yticks(range(len(top_coefficients)), top_coefficients['feature'])
plt.xlabel('Absolute Coefficient Value', fontsize=12)
plt.title('Logistic Regression - Top 15 Feature Coefficients', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3)

# Add legend with corrected interpretation
import matplotlib.patches as mpatches

green_patch = mpatches.Patch(color='green', label='Positive coefficient (higher bad-credit risk)')
red_patch = mpatches.Patch(color='red', label='Negative coefficient (lower bad-credit risk)')
plt.legend(handles=[green_patch, red_patch])

output_path = FIGURES_DIR / 'logistic_regression_coefficients.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Saved figure to: {output_path.resolve()}")
plt.show()
plt.close()

print("Top 10 most influential features:")
display(coefficient_df.head(10))

# Save coefficient data
coefficient_df.to_csv(METRICS_DIR / 'logistic_regression_coefficients.csv', index=False)

# SHAP analysis (if available)
if SHAP_AVAILABLE:
    try:
        print("=== SHAP ANALYSIS FOR RANDOM FOREST ===")
        
        # Get preprocessed training data
        X_train_processed = rf_pipeline.named_steps['preprocessor'].transform(X_train)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(rf_model)
        shap_values = explainer.shap_values(X_train_processed)
        
        # For binary classification, shap_values[1] corresponds to class 1 (bad credit)
        if isinstance(shap_values, list):
            shap_values_class1 = shap_values[1]
        else:
            shap_values_class1 = shap_values
        
        # Create summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values_class1, X_train_processed, 
                        feature_names=feature_names, 
                        show=False, max_display=15)
        plt.title('SHAP Summary Plot - Random Forest', fontsize=14, fontweight='bold')
        
        output_path = FIGURES_DIR / 'shap_summary_plot.png'
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Saved figure to: {output_path.resolve()}")
        plt.show()
        plt.close()
        
    except Exception as e:
        print(f"SHAP analysis failed: {e}")
        print("Skipping SHAP analysis...")
else:
    print("SHAP not available, skipping SHAP analysis...")

In [ ]:
# Final verification: List all saved files
print("=== FIGURES DIRECTORY CONTENTS ===")
for f in FIGURES_DIR.glob("*"):
    print(" -", f.name)

print("\n=== TABLES DIRECTORY CONTENTS ===")
for f in TABLES_DIR.glob("*"):
    print(" -", f.name)

print("\n=== METRICS DIRECTORY CONTENTS ===")
for f in METRICS_DIR.glob("*"):
    print(" -", f.name)

# Generate final summary
print("\n=== FINAL SUMMARY ===")
print()

# Find best performing model
best_model = comparison_df.loc[comparison_df['ROC-AUC'].idxmax()]['Model']
best_auc = comparison_df.loc[comparison_df['ROC-AUC'].idxmax()]['ROC-AUC']
best_accuracy = comparison_df.loc[comparison_df['Accuracy'].idxmax()]['Accuracy']
best_f1 = comparison_df.loc[comparison_df['F1-Score'].idxmax()]['F1-Score']

print(f"Best performing model by ROC-AUC: {best_model} (AUC = {best_auc:.4f})")
print(f"Best accuracy: {comparison_df.loc[comparison_df['Accuracy'].idxmax()]['Model']} ({best_accuracy:.4f})")
print(f"Best F1-Score: {comparison_df.loc[comparison_df['F1-Score'].idxmax()]['Model']} ({best_f1:.4f})")
print()

print("=== PERFORMANCE vs INTERPRETABILITY TRADE-OFF ===")
print()
print("Performance Ranking (by ROC-AUC):")
for i, (_, row) in enumerate(comparison_df.sort_values('ROC-AUC', ascending=False).iterrows()):
    print(f"  {i+1}. {row['Model']}: {row['ROC-AUC']:.4f}")
print()

print("Interpretability Ranking (subjective):")
print("  1. Logistic Regression - Clear coefficient interpretation")
print("  2. Decision Tree - Visual decision rules")
print("  3. Random Forest - Feature importance, ensemble nature")
print("  4. Gradient Boosting - Complex ensemble, less interpretable")
print()

print("=== KEY FINDINGS ===")
print()
print("• Dataset Characteristics:")
print(f"  - {len(df)} samples with {len(df.columns)-1} features")
print(f"  - Class imbalance: {target_percentages[0]:.1f}% good credit, {target_percentages[1]:.1f}% bad credit")
print(f"  - Mixed categorical ({len(categorical_features)}) and numerical ({len(numerical_features)}) features")
print()

print("• Model Performance:")
print(f"  - All models achieved >{comparison_df['Accuracy'].min():.3f} accuracy")
print(f"  - ROC-AUC scores range from {comparison_df['ROC-AUC'].min():.3f} to {comparison_df['ROC-AUC'].max():.3f}")
print(f"  - {best_model} shows the best overall performance")
print()

print("• Important Features (Random Forest):")
top_3_features = feature_importance_df.head(3)['feature'].tolist()
for i, feature in enumerate(top_3_features, 1):
    print(f"  {i}. {feature}")
print()

print("=== LIMITATIONS ===")
print()
print("• Dataset Limitations:")
print("  - Relatively small dataset (1000 samples)")
print("  - Potential bias in historical credit data")
print("  - Some features may not be available in real-time applications")
print()

print("• Methodology Limitations:")
print("  - Limited hyperparameter tuning")
print("  - Single train-test split (though cross-validation was used)")
print("  - No advanced feature engineering techniques")
print()

print("=== FUTURE WORK ===")
print()
print("• Technical Improvements:")
print("  - Hyperparameter optimization with GridSearchCV or RandomizedSearchCV")
print("  - Try additional models (SVM, Neural Networks, XGBoost)")
print("  - Advanced feature engineering and selection")
print("  - Ensemble methods and stacking")
print()

print("• Research Extensions:")
print("  - Fairness analysis and bias mitigation")
print("  - Cost-sensitive learning for different error costs")
print("  - Temporal validation for real-world deployment")
print("  - Comparison with domain-specific credit scoring models")

print("\n" + "="*50)
print(f"ANALYSIS COMPLETE! All results saved to {PROJECT_ROOT / 'results'} directory.")
print("="*50)